# Reinforcement Learning for Global Equity ETF Allocation

This notebook implements a PPO-based RL agent that dynamically allocates across global equity ETFs using macro signals.

**Key Features:**
- **State:** 7 macro signals + BMOM composite per signal ticker
- **Macro signals:** Fed total assets, US M2, TLT price, VIX, yield curve slope (2Y–10Y), S&P 500 earnings, earnings growth
- **Actions:** Allocate to SPY, EWJ, INDA, MCHI, EZU, or BIL (cash)
- **Reward:** Risk-adjusted returns with volatility penalty; discounted expected return = E[r] − λ·σ² (risk-aversion weighted covariance)
- **Momentum:** BMOM composite (weighted multi-lookback) replaces plain 10/20-day momentum
- **Baseline:** BMOM-based ETF rotation for comparison alongside SPY buy-and-hold
- **Algorithm:** Proximal Policy Optimization (PPO)


In [3]:
# %pip install pandas torch

import pandas as pd
import datetime as dt
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import os
from typing import Callable, Dict, Union
from trading_engine.core import (
    read_data, create_model_state, orchestrate_model_backtests,
    orchestrate_model_simulations, orchestrate_portfolio_simulations
)
from trading_engine.models import MODELS
from trading_engine.optimizers import OPTIMIZERS

# RL imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## 1. PPO Policy Network

The policy network maps the macro+momentum state to a probability distribution over ETF allocation actions.


In [4]:
class PolicyNetwork(nn.Module):
    """
    Neural network that maps state features to action probabilities.

    Improvements vs. baseline:
      • LayerNorm after each hidden layer stabilises training with heterogeneous macro inputs.
      • Entropy bonus encourages exploration and prevents premature convergence to a single ETF.
    """

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super(PolicyNetwork, self).__init__()
        self.fc1   = nn.Linear(state_dim, hidden_dim)
        self.ln1   = nn.LayerNorm(hidden_dim)
        self.fc2   = nn.Linear(hidden_dim, hidden_dim)
        self.ln2   = nn.LayerNorm(hidden_dim)
        self.fc3   = nn.Linear(hidden_dim, action_dim)
        self.relu  = nn.ReLU()
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, state):
        x = self.relu(self.ln1(self.fc1(state)))
        x = self.relu(self.ln2(self.fc2(x)))
        action_probs = self.softmax(self.fc3(x))
        return action_probs


class ValueNetwork(nn.Module):
    """
    Neural network that estimates state values for advantage calculation.
    LayerNorm added for symmetry and training stability.
    """

    def __init__(self, state_dim: int, hidden_dim: int = 128):
        super(ValueNetwork, self).__init__()
        self.fc1  = nn.Linear(state_dim, hidden_dim)
        self.ln1  = nn.LayerNorm(hidden_dim)
        self.fc2  = nn.Linear(hidden_dim, hidden_dim)
        self.ln2  = nn.LayerNorm(hidden_dim)
        self.fc3  = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()

    def forward(self, state):
        x = self.relu(self.ln1(self.fc1(state)))
        x = self.relu(self.ln2(self.fc2(x)))
        value = self.fc3(x)
        return value


## 2. PPO Agent

In [5]:
class PPOAgent:
    """
    Proximal Policy Optimization agent for ETF allocation.

    Key change: entropy bonus (−β·H(π)) added to the policy loss to encourage
    exploration and prevent the policy from collapsing to always-BIL or always-SPY.
    """

    def __init__(self, state_dim: int, action_dim: int,
                 lr_policy: float = 3e-4, lr_value: float = 1e-3,
                 gamma: float = 0.95, epsilon_clip: float = 0.2,
                 hidden_dim: int = 128, entropy_coef: float = 0.01):
        self.gamma        = gamma
        self.epsilon_clip = epsilon_clip
        self.action_dim   = action_dim
        self.entropy_coef = entropy_coef  # controls exploration pressure

        # Networks
        self.policy = PolicyNetwork(state_dim, action_dim, hidden_dim)
        self.value  = ValueNetwork(state_dim, hidden_dim)

        # Optimizers
        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=lr_policy)
        self.value_optimizer  = optim.Adam(self.value.parameters(),  lr=lr_value)

        # Trajectory storage
        self.states      = []
        self.actions     = []
        self.rewards     = []
        self.action_probs = []
        self.is_terminals = []

    def select_action(self, state, training=True):
        """Select action based on current policy."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action_probs = self.policy(state_tensor)

        if training:
            dist   = Categorical(action_probs)
            action = dist.sample()
        else:
            action = torch.argmax(action_probs, dim=1)

        return action.item(), action_probs[0, action.item()].item()

    def store_transition(self, state, action, reward, action_prob, is_terminal):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.action_probs.append(action_prob)
        self.is_terminals.append(is_terminal)

    def compute_advantages(self):
        """Compute GAE (Generalized Advantage Estimation)."""
        states_tensor = torch.FloatTensor(np.array(self.states))
        values = self.value(states_tensor).detach().numpy().flatten()

        advantages = []
        returns    = []
        gae        = 0
        next_value = 0

        for t in reversed(range(len(self.rewards))):
            if self.is_terminals[t]:
                next_value = 0
                gae        = 0
            delta = self.rewards[t] + self.gamma * next_value - values[t]
            gae   = delta + self.gamma * 0.95 * gae  # λ = 0.95
            advantages.insert(0, gae)
            returns.insert(0, gae + values[t])
            next_value = values[t]

        advantages = np.array(advantages)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return advantages, np.array(returns)

    def update(self, epochs=10, batch_size=64):
        """Update policy and value networks using PPO with entropy bonus."""
        if len(self.states) == 0:
            return

        advantages, returns = self.compute_advantages()

        states_tensor    = torch.FloatTensor(np.array(self.states))
        actions_tensor   = torch.LongTensor(np.array(self.actions))
        old_probs_tensor = torch.FloatTensor(np.array(self.action_probs))
        advantages_tensor = torch.FloatTensor(advantages)
        returns_tensor   = torch.FloatTensor(returns)

        for _ in range(epochs):
            indices = np.arange(len(self.states))
            np.random.shuffle(indices)

            for start in range(0, len(self.states), batch_size):
                end          = min(start + batch_size, len(self.states))
                batch_idx    = indices[start:end]

                batch_states    = states_tensor[batch_idx]
                batch_actions   = actions_tensor[batch_idx]
                batch_old_probs = old_probs_tensor[batch_idx]
                batch_adv       = advantages_tensor[batch_idx]
                batch_ret       = returns_tensor[batch_idx]

                action_probs = self.policy(batch_states)
                dist         = Categorical(action_probs)
                new_probs    = dist.log_prob(batch_actions).exp()

                ratio = new_probs / (batch_old_probs + 1e-8)

                surr1 = ratio * batch_adv
                surr2 = torch.clamp(ratio, 1 - self.epsilon_clip, 1 + self.epsilon_clip) * batch_adv

                # Entropy bonus: maximising entropy encourages exploration
                entropy     = dist.entropy().mean()
                policy_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy

                value_pred  = self.value(batch_states).squeeze()
                value_loss  = nn.MSELoss()(value_pred, batch_ret)

                self.policy_optimizer.zero_grad()
                policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.policy_optimizer.step()

                self.value_optimizer.zero_grad()
                value_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.value.parameters(), 0.5)
                self.value_optimizer.step()

        self.clear_memory()

    def clear_memory(self):
        self.states       = []
        self.actions      = []
        self.rewards      = []
        self.action_probs = []
        self.is_terminals = []


## 3. RL Model Class for Trading Engine Integration

`RLGlobalEquityAllocator` wraps the PPO agent so it plugs into the existing trading-engine
orchestrator interface.

**Reward design (discounted expected return with risk aversion):**

```
reward_t = r_t  −  λ · σ²_t
```

where `λ` is the `risk_aversion` parameter and `σ²_t` is the rolling portfolio variance.
This is the mean-variance objective: we maximise E[r] while penalising variance, which is
equivalent to a first-order approximation of expected utility for a CARA investor.


In [6]:
class RLGlobalEquityAllocator:
    """
    RL-based allocator for global equity ETFs.

    State = macro features (from load_macro_features) + weighted BMOM per signal ticker.

    Macro features (7):
      fed_assets, us_m2, tlt_price, vix, yield_slope, sp500_earn, earn_growth

    Momentum features (1 per signal ticker):
      Weighted BMOM composite of lookback columns, e.g.:
        0.15*close_momentum_20 + 0.35*close_momentum_60 + 0.35*close_momentum_120 + 0.15*close_momentum_240

    Reward function (mean-variance / risk-aversion objective):
      reward_t = r_t  −  risk_aversion × σ²_t
      where σ²_t = rolling variance of recent portfolio returns.
      This is the scalar approximation to: E[r] − λ · w^T Σ w
      (the full Markowitz-style penalisation uses the covariance matrix Σ).

    state_dim = len(macro_columns) + len(signal_tickers)
    """

    def __init__(
        self,
        trade_tickers: list,
        signal_tickers: list,
        macro_df,
        bmom_columns: list = None,
        bmom_weights: list = None,
        risk_aversion: float = 2.0,   # replaces volatility_penalty; penalises variance
        lr_policy: float = 3e-4,
        lr_value: float = 1e-3,
        hidden_dim: int = 128,
        entropy_coef: float = 0.01,
        trained_agent: PPOAgent = None,
    ):
        self.trade_tickers  = trade_tickers
        self.signal_tickers = signal_tickers
        self.macro_df       = macro_df
        self.macro_columns  = list(macro_df.columns)
        self.bmom_columns   = bmom_columns or []
        self.bmom_weights   = np.array(bmom_weights, dtype=np.float32) if bmom_weights else np.array([])
        self.risk_aversion  = risk_aversion
        self.lr_policy      = lr_policy
        self.lr_value       = lr_value
        self.hidden_dim     = hidden_dim
        self.entropy_coef   = entropy_coef

        self.macro_dim    = len(self.macro_columns)
        self.momentum_dim = len(signal_tickers)
        self.state_dim    = self.macro_dim + self.momentum_dim
        self.action_dim   = len(trade_tickers)

        self.agent      = trained_agent
        self.is_trained = trained_agent is not None

    # ------------------------------------------------------------------
    # Pre-indexing (speed)
    # ------------------------------------------------------------------

    def _build_state_index(self, df: pl.DataFrame) -> dict:
        """
        Pre-compute the BMOM state vector for every date so the training loop
        does a single dict lookup instead of a Polars filter per step.
        Returns {date: np.ndarray of shape (momentum_dim,)}
        """
        index = {}
        if len(self.bmom_columns) == 0 or self.momentum_dim == 0:
            for date in df['date'].unique().to_list():
                index[date] = np.zeros(self.momentum_dim, dtype=np.float32)
            return index

        w = self.bmom_weights
        for date, group in df.group_by('date'):
            feat = np.zeros(self.momentum_dim, dtype=np.float32)
            for i, ticker in enumerate(self.signal_tickers):
                row  = group.filter(pl.col('ticker') == ticker)
                if len(row) == 0:
                    continue
                vals = np.array(
                    [float(row[col][0]) if col in row.columns and row[col][0] is not None else 0.0
                     for col in self.bmom_columns], dtype=np.float32)
                vals     = np.nan_to_num(vals, nan=0.0)
                feat[i]  = float(np.dot(w, vals)) if len(w) == len(vals) else float(vals.mean())
            key        = date[0] if isinstance(date, tuple) else date
            index[key] = feat
        return index

    def _build_price_index(self, prices_df: pl.DataFrame) -> dict:
        index   = {}
        tickers = [t for t in self.trade_tickers if t in prices_df.columns]
        for row in prices_df.iter_rows(named=True):
            d        = row['date']
            index[d] = {t: row[t] for t in tickers if row[t] is not None}
        return index

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _get_macro(self, date) -> np.ndarray:
        if date in self.macro_df.index:
            return self.macro_df.loc[date].values.astype(np.float32)
        past_idx = self.macro_df.index[self.macro_df.index <= date]
        if len(past_idx) > 0:
            return self.macro_df.loc[past_idx[-1]].values.astype(np.float32)
        return np.zeros(self.macro_dim, dtype=np.float32)

    def _get_bmom(self, date_df) -> np.ndarray:
        feat = np.zeros(self.momentum_dim, dtype=np.float32)
        if date_df is None or len(date_df) == 0 or len(self.bmom_columns) == 0:
            return feat
        for i, ticker in enumerate(self.signal_tickers):
            ticker_df = date_df.filter(pl.col('ticker') == ticker)
            if len(ticker_df) == 0:
                continue
            vals = np.zeros(len(self.bmom_columns), dtype=np.float32)
            for j, col in enumerate(self.bmom_columns):
                if col in ticker_df.columns:
                    v       = ticker_df.select(col).to_numpy()[0, 0]
                    vals[j] = 0.0 if np.isnan(v) else float(v)
            feat[i] = float(np.dot(self.bmom_weights, vals)) if len(self.bmom_weights) == len(vals) else float(vals.mean())
        return feat

    def _get_state(self, date, date_df=None) -> np.ndarray:
        return np.concatenate([self._get_macro(date), self._get_bmom(date_df)])

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    def train(
        self,
        model_state: Union[pl.LazyFrame, pl.DataFrame],
        prices: Union[pl.LazyFrame, pl.DataFrame],
        num_episodes: int = 100,
        update_frequency: int = 2000,
        volatility_window: int = 20,
        verbose: bool = True,
    ):
        if verbose:
            print("=" * 80)
            print("TRAINING RL GLOBAL EQUITY ALLOCATOR")
            print("=" * 80)

        if self.agent is None:
            self.agent = PPOAgent(
                self.state_dim, self.action_dim,
                lr_policy=self.lr_policy, lr_value=self.lr_value,
                hidden_dim=self.hidden_dim, entropy_coef=self.entropy_coef,
            )

        df        = model_state.collect() if hasattr(model_state, "collect") else model_state.clone()
        prices_df = prices.collect()      if hasattr(prices, "collect")      else prices.clone()

        dates = df.select('date').unique().sort('date')['date'].to_list()

        if verbose:
            bmom_str = " + ".join(f"{w:.2f}*{c}" for w, c in zip(self.bmom_weights.tolist(), self.bmom_columns)) if self.bmom_columns else "none"
            print(f"Configuration:")
            print(f"  • Training days:   {len(dates)}")
            print(f"  • State dim:       {self.state_dim}  ({self.macro_dim} macro + {self.momentum_dim} bmom)")
            print(f"  • Macro features:  {self.macro_columns}")
            print(f"  • Signal tickers:  {self.signal_tickers}")
            print(f"  • BMOM composite:  {bmom_str}")
            print(f"  • Action space:    {self.action_dim}  →  {self.trade_tickers}")
            print(f"  • Episodes:        {num_episodes}")
            print(f"  • Risk aversion λ: {self.risk_aversion}  (reward = r − λ·σ²)")
            print(f"  • Entropy coef:    {self.entropy_coef}")
            print(f"Pre-indexing data for fast lookup...")

        bmom_index  = self._build_state_index(df)
        price_index = self._build_price_index(prices_df)
        macro_index = {d: self._get_macro(d) for d in dates}

        state_matrix = np.zeros((len(dates), self.state_dim), dtype=np.float32)
        for i, d in enumerate(dates):
            bmom_vec        = bmom_index.get(d, np.zeros(self.momentum_dim, dtype=np.float32))
            state_matrix[i] = np.concatenate([macro_index[d], bmom_vec])

        if verbose:
            print(f"✓ Pre-indexing complete. Starting training...\n")

        episode_rewards = []
        import time
        t0 = time.time()

        for episode in range(num_episodes):
            total_reward   = 0
            timestep       = 0
            recent_returns = []

            for i in range(len(dates) - 1):
                date      = dates[i]
                next_date = dates[i + 1]
                state     = state_matrix[i]

                action_idx, action_prob = self.agent.select_action(state, training=True)
                selected_ticker        = self.trade_tickers[action_idx]

                today = price_index.get(date, {})
                nxt   = price_index.get(next_date, {})

                if selected_ticker in today and selected_ticker in nxt and today[selected_ticker] and nxt[selected_ticker]:
                    daily_return = (nxt[selected_ticker] - today[selected_ticker]) / today[selected_ticker]
                else:
                    daily_return = 0.0

                recent_returns.append(daily_return)
                if len(recent_returns) > volatility_window:
                    recent_returns.pop(0)

                # Reward = E[r] − λ·σ²  (scalar approximation of mean-variance objective)
                # The full version is: E[r] − λ · w^T Σ w where Σ is the covariance matrix.
                # With a single-asset allocation, σ² collapses to the scalar portfolio variance.
                variance = np.var(recent_returns) if len(recent_returns) >= 2 else 0.0
                reward   = daily_return - self.risk_aversion * variance

                is_terminal = (i == len(dates) - 2)
                self.agent.store_transition(state, action_idx, reward, action_prob, is_terminal)

                total_reward += daily_return
                timestep     += 1

                if timestep % update_frequency == 0:
                    self.agent.update()

            self.agent.update()
            episode_rewards.append(total_reward)

            if verbose and (episode + 1) % 10 == 0:
                elapsed   = time.time() - t0
                per_ep    = elapsed / (episode + 1)
                remaining = per_ep * (num_episodes - episode - 1)
                print(f"Episode {episode+1}/{num_episodes}  |  Avg Return (last 10): {np.mean(episode_rewards[-10:]):.4f}  |  ETA: {remaining/60:.1f} min")

        self.is_trained = True

        if verbose:
            total_time = time.time() - t0
            print(f"✓ Training complete in {total_time/60:.1f} min!")
            print("=" * 80)
            plt.figure(figsize=(10, 4))
            plt.plot(episode_rewards)
            plt.title('Training Progress: Episode Returns')
            plt.xlabel('Episode')
            plt.ylabel('Total Return')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    # ------------------------------------------------------------------
    # Save / Load
    # ------------------------------------------------------------------

    def save(self, path: str):
        if not self.is_trained:
            raise ValueError("Model must be trained before saving")
        torch.save({
            'policy_state_dict': self.agent.policy.state_dict(),
            'value_state_dict':  self.agent.value.state_dict(),
            'trade_tickers':     self.trade_tickers,
            'signal_tickers':    self.signal_tickers,
            'macro_columns':     self.macro_columns,
            'bmom_columns':      self.bmom_columns,
            'bmom_weights':      self.bmom_weights.tolist(),
            'risk_aversion':     self.risk_aversion,
            'lr_policy':         self.lr_policy,
            'lr_value':          self.lr_value,
            'hidden_dim':        self.hidden_dim,
            'entropy_coef':      self.entropy_coef,
            'state_dim':         self.state_dim,
            'action_dim':        self.action_dim,
        }, f'{path}.pth')
        print(f"✓ Model saved to {path}.pth")

    @classmethod
    def load(cls, path: str, macro_df) -> 'RLGlobalEquityAllocator':
        checkpoint = torch.load(f'{path}.pth')
        allocator  = cls(
            trade_tickers=checkpoint['trade_tickers'],
            signal_tickers=checkpoint['signal_tickers'],
            macro_df=macro_df,
            bmom_columns=checkpoint.get('bmom_columns', []),
            bmom_weights=checkpoint.get('bmom_weights', []),
            risk_aversion=checkpoint.get('risk_aversion', 2.0),
            lr_policy=checkpoint['lr_policy'],
            lr_value=checkpoint['lr_value'],
            hidden_dim=checkpoint['hidden_dim'],
            entropy_coef=checkpoint.get('entropy_coef', 0.01),
        )
        allocator.agent = PPOAgent(
            checkpoint['state_dim'], checkpoint['action_dim'],
            lr_policy=checkpoint['lr_policy'], lr_value=checkpoint['lr_value'],
            hidden_dim=checkpoint['hidden_dim'],
            entropy_coef=checkpoint.get('entropy_coef', 0.01),
        )
        allocator.agent.policy.load_state_dict(checkpoint['policy_state_dict'])
        allocator.agent.value.load_state_dict(checkpoint['value_state_dict'])
        allocator.is_trained = True
        print(f"✓ Model loaded from {path}.pth")
        return allocator

    # ------------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------------

    def __call__(self, lf: pl.LazyFrame) -> pl.LazyFrame:
        if not self.is_trained:
            raise ValueError("Model must be trained before generating signals")

        df         = lf.collect()
        dates      = df.select('date').unique().sort('date')['date'].to_list()
        bmom_index = self._build_state_index(df)
        weights_list = []

        for date in dates:
            bmom_vec   = bmom_index.get(date, np.zeros(self.momentum_dim, dtype=np.float32))
            state      = np.concatenate([self._get_macro(date), bmom_vec])
            action_idx, _ = self.agent.select_action(state, training=False)
            weight_dict   = {'date': date}
            for i, ticker in enumerate(self.trade_tickers):
                weight_dict[ticker] = 1.0 if i == action_idx else 0.0
            weights_list.append(weight_dict)

        return pl.DataFrame(weights_list).lazy()


print("✓ RLGlobalEquityAllocator class defined!")


✓ RLGlobalEquityAllocator class defined!


## 4. Configuration and Data Loading

### BMOM — Blended Momentum Composite

Instead of a single 10-day or 20-day lookback, we use **BMOM**: a weighted blend across
multiple horizons (20, 60, 120, 240 days). This captures:

- **Short-term reversal / noise filtering** — the 20-day anchors recent moves
- **Trend persistence** — the 60 and 120-day horizons capture intermediate trends
- **Regime stickiness** — the 240-day horizon tracks secular risk-on/risk-off cycles

The weights (0.15 / 0.35 / 0.35 / 0.15) are symmetric and down-weight the extremes,
similar to a triangular filter.

BMOM is used for **both** the signal features fed to the RL agent (TLT, VIXY regime signals)
**and** the standalone BMOM Baseline that we benchmark against.


In [7]:
# ── Universe ──────────────────────────────────────────────────────────────────
universe = [
    "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
    "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
    "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
    "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US", "EWJ-US"
]

# ── Trade tickers — the 6 actions the RL agent can select ────────────────────
# SPY  : US large-cap equities (core risk-on)
# EWJ  : Japan equities (developed market diversifier)
# INDA : India equities (EM growth)
# MCHI : China equities (EM value / risk)
# EZU  : Eurozone equities (developed Europe)
# BIL  : T-bill ETF (cash / risk-off)
trade_tickers = ["SPY-US", "EWJ-US", "INDA-US", "MCHI-US", "EZU-US", "BIL-US"]

# ── Signal tickers — their BMOM scores are part of the RL state ──────────────
# TLT  : Treasury bond ETF — a *risk-off barometer*.
#         Rising TLT BMOM → flight to safety → RL should favour BIL or defensives.
# VIXY : VIX futures ETF — *fear gauge*.
#         Rising VIXY BMOM → elevated volatility regime → RL should reduce equity risk.
# IBIT : Bitcoin ETF — *crypto/risk-appetite signal*.
#         Rising IBIT BMOM → broad risk-on sentiment, especially in EM and growth.
signal_tickers = ["TLT-US", "VIXY-US", "IBIT-US"]

# ── BMOM parameters ──────────────────────────────────────────────────────────
# BMOM = 0.15*mom20 + 0.35*mom60 + 0.35*mom120 + 0.15*mom240
# Triangular-style weighting: down-weights noisy short-term and very slow long-term,
# emphasises 1-6 month trend persistence — the empirically strongest momentum horizon.
bmom_columns = ["close_momentum_20", "close_momentum_60", "close_momentum_120", "close_momentum_240"]
bmom_weights = [0.15, 0.35, 0.35, 0.15]

# ── Data config ───────────────────────────────────────────────────────────────
DATA_DIR      = "data"
initial_value = 1_000_000
start_date    = dt.date(2004, 1, 2)
end_date      = dt.date(2025, 9, 10)


## 5. Macro Features Loading

### Financial Intuition for Each Signal

| Feature | Source | Frequency | Why it matters |
|---|---|---|---|
| **fed_assets** | WALCL (FRED) | Weekly | Fed balance sheet expansion (QE) = risk-on, liquidity tailwind for equities |
| **us_m2** | M2 (FRED) | Weekly | Money supply growth drives nominal asset prices and inflation expectations |
| **tlt_price** | TLT-US ETF | Daily | Long bond price captures duration risk appetite and interest rate regime |
| **vix** | VIXY-US ETF | Daily | Equity volatility regime — high VIX → de-risk; low VIX → risk-on |
| **yield_slope** | 10Y − 2Y (FRED) | Daily | Yield curve slope: steep = expansion, inverted = recession risk |
| **sp500_earn** | Quarterly EPS | Quarterly | Absolute earnings level drives equity valuation and PE multiples |
| **earn_growth** | QoQ EPS growth | Quarterly | Earnings momentum: accelerating growth = equity tailwind |

All features are normalised via 252-day rolling z-score so the neural network
sees stationary, comparably scaled inputs regardless of level.


In [8]:
def load_macro_features(
    data_dir: str,
    start_date: dt.date,
    end_date: dt.date,
    prices_df=None,
) -> pd.DataFrame:
    """
    Load and preprocess macro signals into a daily pd.DataFrame.

    Features built:
      1. fed_assets  — Fed total assets (WALCL), weekly → forward-filled
      2. us_m2       — US M2 money supply, weekly → forward-filled
      3. tlt_price   — TLT-US ETF close price, daily (from prices_df)
      4. vix         — VIXY-US close price (vol proxy), daily (from prices_df)
      5. yield_slope — 10Y minus 2Y Treasury spread, daily
      6. sp500_earn  — S&P 500 earnings level, quarterly → forward-filled
      7. earn_growth — S&P 500 earnings growth, quarterly → forward-filled

    All features are forward-filled to business-day frequency and normalized
    via 252-day rolling z-score (min 30 obs).
    """
    def load_csv(path, date_col):
        """Load a FRED-style CSV, coercing all value columns to float."""
        df = pd.read_csv(path, parse_dates=[date_col], index_col=date_col)
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        return df

    date_range = pd.date_range(start=start_date, end=end_date, freq='B')
    macro_df = pd.DataFrame(index=date_range)

    # 1. Fed total assets (weekly)
    fed_path = os.path.join(data_dir, 'fed_total_assets.csv')
    if os.path.exists(fed_path):
        fed = load_csv(fed_path, 'observation_date')
        macro_df = macro_df.join(fed[['WALCL']].rename(columns={'WALCL': 'fed_assets'}), how='left')
    else:
        print(f"  ⚠ Missing: {fed_path}")

    # 2. US M2 (weekly — use first data column)
    m2_path = os.path.join(data_dir, 'USM2.csv')
    if os.path.exists(m2_path):
        m2 = load_csv(m2_path, 'observation_date')
        m2_col = m2.columns[0]
        macro_df = macro_df.join(m2[[m2_col]].rename(columns={m2_col: 'us_m2'}), how='left')
    else:
        print(f"  ⚠ Missing: {m2_path}")

    # 3 & 4. TLT and VIXY from prices_df (daily)
    if prices_df is not None:
        prices_pd = prices_df.collect().to_pandas() if hasattr(prices_df, 'collect') else prices_df.to_pandas()
        prices_pd['date'] = pd.to_datetime(prices_pd['date'])
        prices_pd = prices_pd.set_index('date')

        if 'TLT-US' in prices_pd.columns:
            macro_df = macro_df.join(
                pd.to_numeric(prices_pd['TLT-US'], errors='coerce').rename('tlt_price'), how='left'
            )
        else:
            print("  ⚠ TLT-US not found in prices_df")

        if 'VIXY-US' in prices_pd.columns:
            macro_df = macro_df.join(
                pd.to_numeric(prices_pd['VIXY-US'], errors='coerce').rename('vix'), how='left'
            )
        else:
            print("  ⚠ VIXY-US not found in prices_df")
    else:
        print("  ⚠ prices_df not provided — TLT and VIX features skipped")

    # 5. Yield curve slope 10Y-2Y (daily)
    slope_path = os.path.join(data_dir, 'slope_10y_2y.csv')
    if os.path.exists(slope_path):
        slope = load_csv(slope_path, 'observation_date')
        macro_df = macro_df.join(slope[['T10Y2Y']].rename(columns={'T10Y2Y': 'yield_slope'}), how='left')
    else:
        print(f"  ⚠ Missing: {slope_path}")

    # 6 & 7. S&P 500 earnings & earnings growth (quarterly)
    earnings_path = os.path.join(data_dir, 'sp500_quaterly_growth.csv')
    if os.path.exists(earnings_path):
        earnings = load_csv(earnings_path, 'observation_date')
        cols = earnings.columns.tolist()
        if len(cols) >= 1:
            macro_df = macro_df.join(earnings[[cols[0]]].rename(columns={cols[0]: 'sp500_earn'}), how='left')
        if len(cols) >= 2:
            macro_df = macro_df.join(earnings[[cols[1]]].rename(columns={cols[1]: 'earn_growth'}), how='left')
    else:
        print(f"  ⚠ Missing: {earnings_path}")

    # Forward-fill low-frequency gaps, then back-fill any leading NaNs
    macro_df = macro_df.ffill().bfill()

    # Ensure float64 throughout before rolling
    macro_df = macro_df.apply(pd.to_numeric, errors='coerce').ffill().bfill()

    # Normalize: 252-day rolling z-score
    for col in macro_df.columns:
        roll = macro_df[col].rolling(window=252, min_periods=30)
        macro_df[col] = (macro_df[col] - roll.mean()) / (roll.std() + 1e-8)
    macro_df = macro_df.fillna(0.0)

    # Convert DatetimeIndex → date objects for lookup alongside pl.Date values
    macro_df.index = macro_df.index.date

    print(f"✓ Macro features loaded: {list(macro_df.columns)}")
    print(f"  Shape: {macro_df.shape}  |  {macro_df.index[0]} → {macro_df.index[-1]}")
    return macro_df

In [9]:
# Load market data with all bmom lookback columns
lf = read_data()
model_state, prices = create_model_state(
    lf=lf,
    features=bmom_columns,
    start_date=start_date,
    end_date=end_date,
    universe=universe,
)
print("Market data loaded!")

# Build macro features DataFrame (prices_df provides TLT and VIXY)
macro_df = load_macro_features(DATA_DIR, start_date, end_date, prices_df=prices)
macro_df.head()

Market data loaded!
✓ Macro features loaded: ['fed_assets', 'us_m2', 'tlt_price', 'vix', 'yield_slope', 'sp500_earn', 'earn_growth']
  Shape: (5659, 7)  |  2004-01-02 → 2025-09-10


,fed_assets,us_m2,tlt_price,vix,yield_slope,sp500_earn,earn_growth
2004-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2004-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2004-01-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2004-01-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2004-01-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 9. Financial Intuition White Paper

### Why These Signals?

#### Macro Regime Signals

**Federal Reserve Total Assets (WALCL)**
The Fed balance sheet is the single most important macro variable for global equities since 2008.
Quantitative easing (balance sheet expansion) injects reserves into the financial system, compressing
risk premia and pushing capital into risk assets. When the Fed is expanding (positive z-score),
equities — especially EM — tend to outperform. QT (contraction) reverses this. The signal has a
~1–3 month lead on equity market moves.

**US M2 Money Supply**
M2 growth tracks the availability of credit and purchasing power in the real economy. Accelerating
M2 supports nominal asset prices (equities and commodities) and reduces real rates. Decelerating
M2 — particularly when it goes negative as in 2022 — is a leading indicator of equity drawdowns.

**TLT Price (Long Bond ETF)**
Long bonds and equities compete for capital. A rising TLT signals falling long yields, which
lowers the discount rate applied to future earnings — particularly bullish for growth equities
and EM. A falling TLT (rising yields) compresses equity multiples. We use TLT's price level
(z-scored) rather than raw yield so the neural network sees a normalised, momentum-compatible input.

**VIX (VIXY Proxy)**
Volatility is the canonical risk-off indicator. When implied volatility spikes, institutions
de-risk, correlations rise toward 1, and even fundamentally sound markets sell off. We use
VIXY (VIX futures ETF) rather than spot VIX because it is tradeable and experiences the
same daily prices as the trade universe.

**Yield Curve Slope (10Y − 2Y)**
An inverted yield curve (negative slope) has preceded every US recession since 1960. It signals
that markets expect future short rates to fall — i.e., economic weakness. A steep curve signals
expansion. We include the raw spread (not z-scored inside the macro normalisation step to preserve
sign information, but z-scored with the rest before input to the network).

**S&P 500 Earnings & Earnings Growth**
Earnings are the fundamental anchor of equity valuations. Rising absolute earnings justify higher
price levels (PE expansion is less required). Accelerating earnings growth is a classic growth
signal that attracts capital to the US equity market. These quarterly series are forward-filled
to daily frequency — they act as slow-moving regime indicators rather than fast signals.

---

#### Momentum Signals: Why BMOM?

Plain short-window momentum (10-day or 20-day) is noisy and prone to mean-reversion. Academic
literature on cross-sectional momentum (Jegadeesh & Titman 1993, Asness et al. 2013) shows that
the strongest risk-adjusted momentum signal spans roughly 2–12 months. BMOM blends four horizons:

| Window | Weight | Rationale |
|--------|--------|-----------|
| 20d    | 0.15   | Short-term noise filter |
| 60d    | 0.35   | 3-month trend (quarterly cycle) |
| 120d   | 0.35   | 6-month trend (intermediate cycle) |
| 240d   | 0.15   | 12-month trend (secular / factor cycle) |

The triangular weighting down-weights the extremes and emphasises 3–6 month momentum —
the horizon with the best empirical Sharpe ratio net of turnover.

Applied to **TLT and VIXY** specifically:
- TLT BMOM > 0 → sustained bond rally → risk-off regime → favour BIL or EZU
- TLT BMOM < 0 → bonds under pressure → reflationary regime → favour SPY, INDA
- VIXY BMOM > 0 → elevated vol regime → reduce equity allocation
- IBIT BMOM > 0 → crypto risk appetite → broad EM and growth tailwind (EWJ, INDA, SPY)

---

#### Reward Design: Risk-Aversion vs. Volatility Penalty

The original reward `r_t − λ·σ_t` (penalising standard deviation) is inconsistent with
standard portfolio theory. We use:

```
reward_t = r_t − λ · σ²_t
```

Penalising **variance** (not std) is the correct form. This is the scalar projection of the
mean-variance objective `E[r] − λ · w^T Σ w`, which underpins CAPM, the Black-Litterman model,
and all classical risk-adjusted portfolio construction. `λ = 2.0` corresponds to moderate
risk aversion (equivalent to a CARA investor with relative risk aversion ≈ 2).


In [ ]:
# ── Walk-Forward Validation: Train 2004–2010, Test 2011–2025 ─────────────────
import datetime as dt

TRAIN_END   = dt.date(2010, 12, 31)
TEST_START  = dt.date(2011, 1, 1)

# ── 1. Split model_state and prices ──────────────────────────────────────────
ms_pd = (model_state.collect() if hasattr(model_state, "collect") else model_state).to_pandas()
ms_pd["date"] = pd.to_datetime(ms_pd["date"]).dt.date

ms_train = pl.from_pandas(ms_pd[ms_pd["date"] <= TRAIN_END].copy())
ms_test  = pl.from_pandas(ms_pd[ms_pd["date"] >= TEST_START].copy())

prices_pd = (prices.collect() if hasattr(prices, "collect") else prices).to_pandas()
prices_pd["date"] = pd.to_datetime(prices_pd["date"]).dt.date

prices_train = pl.from_pandas(prices_pd[prices_pd["date"] <= TRAIN_END].copy())
prices_test  = pl.from_pandas(prices_pd[prices_pd["date"] >= TEST_START].copy())

# ── 2. Load macro features for each split ────────────────────────────────────
macro_train = load_macro_features(DATA_DIR, start_date,  TRAIN_END,  prices_df=prices)
macro_test  = load_macro_features(DATA_DIR, TEST_START,  end_date,   prices_df=prices)
# Note: we pass full prices to macro loader (TLT/VIXY extraction) but
# the allocator only ever looks up dates within its own split.

# ── 3. Train ONLY on 2004–2010 ───────────────────────────────────────────────
wf_allocator = RLGlobalEquityAllocator(
    trade_tickers=trade_tickers,
    signal_tickers=signal_tickers,
    macro_df=macro_train,          # ← training macro only
    bmom_columns=bmom_columns,
    bmom_weights=bmom_weights,
    risk_aversion=2.0,
    lr_policy=3e-4,
    lr_value=1e-3,
    hidden_dim=128,
    entropy_coef=0.01,
)

wf_allocator.train(
    model_state=ms_train,          # ← training data only
    prices=prices_train,
    num_episodes=100,
    update_frequency=500,          # shorter update window (fewer training days)
    verbose=True,
)

✓ Macro features loaded: ['fed_assets', 'us_m2', 'tlt_price', 'vix', 'yield_slope', 'sp500_earn', 'earn_growth']
  Shape: (1826, 7)  |  2004-01-02 → 2010-12-31
✓ Macro features loaded: ['fed_assets', 'us_m2', 'tlt_price', 'vix', 'yield_slope', 'sp500_earn', 'earn_growth']
  Shape: (3833, 7)  |  2011-01-03 → 2025-09-10
TRAINING RL GLOBAL EQUITY ALLOCATOR
Configuration:
  • Training days:   2078
  • State dim:       10  (7 macro + 3 bmom)
  • Macro features:  ['fed_assets', 'us_m2', 'tlt_price', 'vix', 'yield_slope', 'sp500_earn', 'earn_growth']
  • Signal tickers:  ['TLT-US', 'VIXY-US', 'IBIT-US']
  • BMOM composite:  0.15*close_momentum_20 + 0.35*close_momentum_60 + 0.35*close_momentum_120 + 0.15*close_momentum_240
  • Action space:    6  →  ['SPY-US', 'EWJ-US', 'INDA-US', 'MCHI-US', 'EZU-US', 'BIL-US']
  • Episodes:        100
  • Risk aversion λ: 2.0  (reward = r − λ·σ²)
  • Entropy coef:    0.01
Pre-indexing data for fast lookup...
✓ Pre-indexing complete. Starting training...



In [ ]:
# ── 4. Run OOS inference on 2011–2025 ────────────────────────────────────────
# Swap in the test macro so the agent can look up post-2010 dates
wf_allocator.macro_df = macro_test

oos_weights = wf_allocator(ms_test.lazy())   # calls __call__ → inference only

# ── 5. Compute OOS returns ───────────────────────────────────────────────────
oos_w = oos_weights.collect().to_pandas()
oos_w["date"] = pd.to_datetime(oos_w["date"])
oos_w = oos_w.set_index("date").sort_index()

p_test = prices_pd[prices_pd["date"] >= TEST_START].copy()
p_test["date"] = pd.to_datetime(p_test["date"])
p_test = p_test.set_index("date").apply(pd.to_numeric, errors="coerce")

# Align dates
common = oos_w.index.intersection(p_test.index)
w_aligned = oos_w.reindex(common)[trade_tickers].fillna(0.0)
p_aligned = p_test.reindex(common)[trade_tickers].ffill()

oos_ret   = (w_aligned.shift(1) * p_aligned.pct_change().fillna(0.0)).sum(axis=1)
spy_oos   = p_aligned["SPY-US"].pct_change().fillna(0.0)

# BMOM baseline OOS
ms_test_pd = ms_pd[ms_pd["date"] >= TEST_START].copy()
ms_test_pd["date"] = pd.to_datetime(ms_test_pd["date"])
ms_trade_oos = ms_test_pd[ms_test_pd["ticker"].isin(trade_tickers)].copy()
for col in bmom_columns:
    ms_trade_oos[col] = pd.to_numeric(ms_trade_oos[col], errors="coerce").fillna(0.0)
ms_trade_oos["bmom"] = ms_trade_oos[bmom_columns].values @ np.array(bmom_weights)
bmom_wide_oos = ms_trade_oos.pivot(index="date", columns="ticker", values="bmom")
bmom_wide_oos.index = pd.to_datetime(bmom_wide_oos.index)
bmom_wide_oos = bmom_wide_oos.reindex(common).ffill().bfill()
best_oos = bmom_wide_oos.dropna(how="all").idxmax(axis=1)
bmom_w_oos = pd.DataFrame(0.0, index=common, columns=trade_tickers)
for date, ticker in best_oos.items():
    if ticker in bmom_w_oos.columns:
        bmom_w_oos.loc[date, ticker] = 1.0
bmom_ret_oos = (bmom_w_oos.shift(1) * p_aligned.pct_change().fillna(0.0)).sum(axis=1)

# ── 6. Performance stats ──────────────────────────────────────────────────────
def perf_stats(ret, label):
    ann = ret.mean() * 252
    vol = ret.std()  * np.sqrt(252)
    sr  = ann / vol if vol > 0 else 0
    dd  = (1 + ret).cumprod()
    dd  = (dd / dd.cummax() - 1).min()
    return {"Strategy": label, "Ann. Return": f"{ann:.1%}",
            "Ann. Vol": f"{vol:.1%}", "Sharpe": f"{sr:.2f}", "Max DD": f"{dd:.1%}"}

summary = pd.DataFrame([
    perf_stats(oos_ret,     "RL Model (OOS 2011–2025)"),
    perf_stats(bmom_ret_oos,"BMOM Baseline (OOS)"),
    perf_stats(spy_oos,     "SPY Buy & Hold (OOS)"),
]).set_index("Strategy")

print("=" * 55)
print("OUT-OF-SAMPLE PERFORMANCE  (trained on 2004–2010 only)")
print("=" * 55)
print(summary.to_string())

# ── 7. Correlation with SPY ───────────────────────────────────────────────────
corr_oos = pd.DataFrame({
    "RL Model": oos_ret, "BMOM": bmom_ret_oos, "SPY": spy_oos
}).corr()
print("\nOOS Return Correlations:")
print(corr_oos.round(3).to_string())

# ── 8. Plot ───────────────────────────────────────────────────────────────────
rl_nav   = (1 + oos_ret).cumprod()
bmom_nav = (1 + bmom_ret_oos).cumprod()
spy_nav  = (1 + spy_oos).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [3, 1]})

ax = axes[0]
ax.plot(rl_nav.index,   rl_nav.values,   label="RL Model (OOS)",  linewidth=2,   color="#2196F3")
ax.plot(bmom_nav.index, bmom_nav.values, label="BMOM Baseline",   linewidth=2,   color="#FF9800", linestyle="--")
ax.plot(spy_nav.index,  spy_nav.values,  label="SPY Buy & Hold",  linewidth=1.5, color="#9E9E9E", linestyle=":")
ax.axvspan(pd.Timestamp("2020-02-01"), pd.Timestamp("2020-04-30"),
           alpha=0.15, color="red", label="COVID crash")
ax.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2022-12-31"),
           alpha=0.15, color="orange", label="Rate hike cycle")
ax.set_title("Out-of-Sample: RL Global Equity Allocator\n(Trained 2004–2010 | Tested 2011–2025)",
             fontsize=14, fontweight="bold")
ax.set_ylabel("NAV (log scale)")
ax.legend(loc="upper left", fontsize=9)
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

ax2 = axes[1]
im  = ax2.imshow(corr_oos.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax2.set_xticks(range(len(corr_oos.columns)))
ax2.set_yticks(range(len(corr_oos.index)))
ax2.set_xticklabels(corr_oos.columns)
ax2.set_yticklabels(corr_oos.index)
for i in range(len(corr_oos.index)):
    for j in range(len(corr_oos.columns)):
        ax2.text(j, i, f"{corr_oos.iloc[i,j]:.2f}", ha="center", va="center",
                 fontsize=12, fontweight="bold",
                 color="white" if abs(corr_oos.iloc[i,j]) > 0.6 else "black")
ax2.set_title("OOS Return Correlation Matrix", fontsize=11)
plt.colorbar(im, ax=ax2, fraction=0.02, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
# ── Drawdown-Differential Regime Switch Overlay ───────────────────────────────
# Logic:
#   - Track rolling drawdown of RL and BMOM separately
#   - If (RL_drawdown - BMOM_drawdown) < -DD_THRESHOLD, switch to BMOM
#   - Stay in BMOM until RL recovers to within RECOVERY_THRESHOLD of its peak
#   - Optionally blend (alpha) instead of hard switch for smoother transitions

DD_THRESHOLD       = 0.08   # switch to BMOM if RL is 8% deeper in drawdown than BMOM
RECOVERY_THRESHOLD = 0.03   # return to RL once RL drawdown is within 3% of BMOM
DD_WINDOW          = 60     # rolling window (days) to compute drawdown from local peak
BLEND_ALPHA        = 0.0    # 0.0 = hard switch, 0.5 = 50/50 blend when switching

# ── Compute rolling drawdowns ─────────────────────────────────────────────────
def rolling_drawdown(ret_series, window=DD_WINDOW):
    """Max drawdown from rolling peak over the last `window` days."""
    nav   = (1 + ret_series).cumprod()
    peaks = nav.rolling(window, min_periods=1).max()
    dd    = (nav / peaks) - 1        # always <= 0
    return dd

rl_dd   = rolling_drawdown(oos_ret)
bmom_dd = rolling_drawdown(bmom_ret_oos)

dd_diff = rl_dd - bmom_dd            # negative when RL is deeper in drawdown than BMOM

# ── Build regime signal ───────────────────────────────────────────────────────
# 0 = use RL weights,  1 = use BMOM weights
regime = pd.Series(0.0, index=oos_ret.index)
in_bmom_regime = False

for date in oos_ret.index:
    if not in_bmom_regime:
        # Switch to BMOM if RL drawdown exceeds BMOM drawdown by threshold
        if dd_diff.loc[date] < -DD_THRESHOLD:
            in_bmom_regime = True
    else:
        # Return to RL once drawdown differential recovers
        if dd_diff.loc[date] > -RECOVERY_THRESHOLD:
            in_bmom_regime = False

    regime.loc[date] = 1.0 if in_bmom_regime else 0.0

# ── Blend returns ─────────────────────────────────────────────────────────────
# When regime=1 (BMOM): weight = (1-alpha)*BMOM + alpha*RL
# When regime=0 (RL):   pure RL
rl_weight   = (1 - regime) + regime * BLEND_ALPHA
bmom_weight = regime * (1 - BLEND_ALPHA)

hybrid_ret  = rl_weight * oos_ret + bmom_weight * bmom_ret_oos

# ── Performance comparison ────────────────────────────────────────────────────
summary_hybrid = pd.DataFrame([
    perf_stats(hybrid_ret,   "Hybrid (RL + DD Switch)"),
    perf_stats(oos_ret,      "RL Model (OOS)"),
    perf_stats(bmom_ret_oos, "BMOM Baseline (OOS)"),
    perf_stats(spy_oos,      "SPY Buy & Hold (OOS)"),
]).set_index("Strategy")

print("=" * 60)
print("HYBRID STRATEGY PERFORMANCE  (OOS 2011–2025)")
print("=" * 60)
print(summary_hybrid.to_string())

pct_in_bmom = regime.mean() * 100
print(f"\nTime spent in BMOM regime: {pct_in_bmom:.1f}%")
print(f"Time spent in RL regime:   {100 - pct_in_bmom:.1f}%")

# ── Plot ──────────────────────────────────────────────────────────────────────

# Force everything to a clean shared DatetimeIndex
idx = pd.DatetimeIndex(oos_ret.index)

hybrid_ret_p  = hybrid_ret.copy();    hybrid_ret_p.index  = idx
oos_ret_p     = oos_ret.copy();       oos_ret_p.index     = idx
bmom_ret_p    = bmom_ret_oos.copy();  bmom_ret_p.index    = idx
spy_ret_p     = spy_oos.copy();       spy_ret_p.index     = idx
dd_diff_p     = dd_diff.copy();       dd_diff_p.index     = idx
regime_p      = regime.copy();        regime_p.index      = idx

hybrid_nav = (1 + hybrid_ret_p).cumprod()
rl_nav     = (1 + oos_ret_p).cumprod()
bmom_nav   = (1 + bmom_ret_p).cumprod()
spy_nav    = (1 + spy_ret_p).cumprod()

fig, axes = plt.subplots(3, 1, figsize=(14, 14),
                         gridspec_kw={"height_ratios": [3, 1, 1]})

# ── Top: NAV curves ───────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(hybrid_nav.index, hybrid_nav.values, label="Hybrid (RL + DD Switch)",
        linewidth=2.5, color="#4CAF50")
ax.plot(rl_nav.index,     rl_nav.values,     label="RL Model (OOS)",
        linewidth=1.5, color="#2196F3", alpha=0.6, linestyle="--")
ax.plot(bmom_nav.index,   bmom_nav.values,   label="BMOM Baseline",
        linewidth=1.5, color="#FF9800", alpha=0.6, linestyle="--")
ax.plot(spy_nav.index,    spy_nav.values,    label="SPY Buy & Hold",
        linewidth=1.2, color="#9E9E9E", linestyle=":")

# Shade BMOM regime periods
in_regime   = False
start_shade = None
for date, val in regime_p.items():
    if val == 1.0 and not in_regime:
        start_shade = date
        in_regime   = True
    elif val == 0.0 and in_regime:
        ax.axvspan(start_shade, date, alpha=0.12, color="orange", label="_nolegend_")
        in_regime   = False
if in_regime:
    ax.axvspan(start_shade, regime_p.index[-1], alpha=0.12, color="orange")
ax.axvspan(pd.Timestamp("1900-01-01"), pd.Timestamp("1900-01-02"),
           alpha=0.25, color="orange", label="BMOM regime active")

ax.set_title("Hybrid Strategy: RL + Drawdown-Differential Regime Switch\n"
             f"(Switch threshold: {DD_THRESHOLD:.0%} | Recovery: {RECOVERY_THRESHOLD:.0%})",
             fontsize=13, fontweight="bold")
ax.set_ylabel("NAV (log scale)")
ax.set_yscale("log")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)

# ── Middle: drawdown differential ────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(dd_diff_p.index, dd_diff_p.values * 100, color="#9C27B0", linewidth=1.2)
ax2.axhline(-DD_THRESHOLD * 100,       color="red",   linestyle="--", linewidth=1,
            label=f"Switch threshold (−{DD_THRESHOLD:.0%})")
ax2.axhline(-RECOVERY_THRESHOLD * 100, color="green", linestyle="--", linewidth=1,
            label=f"Recovery threshold (−{RECOVERY_THRESHOLD:.0%})")
ax2.fill_between(dd_diff_p.index, dd_diff_p.values * 100, 0,
                 where=(dd_diff_p.values < -DD_THRESHOLD),
                 alpha=0.3, color="red", label="In BMOM regime")
ax2.set_ylabel("RL DD − BMOM DD (%)")
ax2.set_title("Drawdown Differential (RL minus BMOM)", fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ── Bottom: regime indicator ──────────────────────────────────────────────────
ax3 = axes[2]
ax3.fill_between(regime_p.index, regime_p.values,     alpha=0.6, color="orange", label="BMOM regime")
ax3.fill_between(regime_p.index, 1 - regime_p.values, alpha=0.4, color="#2196F3", label="RL regime")
ax3.set_yticks([0, 1])
ax3.set_yticklabels(["RL", "BMOM"])
ax3.set_title("Active Regime", fontsize=10)
ax3.legend(fontsize=8, loc="upper right")
ax3.grid(True, alpha=0.2)

for ax_ in axes:
    ax_.set_xlim(idx[0], idx[-1])

plt.tight_layout()
plt.show()


# Portfolio Integration Analysis — RL Global Equity Allocator

Full marginal contribution to risk, diversification metrics, blended portfolio scenarios, drawdown analysis, and rolling metrics.

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import polars as pl
from matplotlib.ticker import FuncFormatter
from trading_engine.core import (
    read_data, create_model_state,
    orchestrate_model_backtests, orchestrate_model_simulations,
    orchestrate_portfolio_aggregation, orchestrate_portfolio_optimizations,
    orchestrate_portfolio_simulations,
)
from trading_engine.model_state import FEATURES
from trading_engine.models import MODELS

# ── Config (matches paper.yaml) ──────────────────────────────
_universe = [
    "TLT-US", "IEI-US", "SHY-US", "BIL-US", "SLV-US", "GLD-US",
    "USO-US", "UNG-US", "SPY-US", "EWJ-US", "INDA-US", "MCHI-US",
    "IBIT-US", "ETHA-US", "EZU-US", "VIXY-US",
]
_start       = datetime.date(2011, 1, 1)
_end         = datetime.date(2025, 1, 1)
_aggregators = ["model_mvo_amma_constrained"]
_optimizers  = ["mean_variance_constrained"]
_init_val    = 1_000_000
_prod_models = [
    "TLT_AMMA", "IEI_AMMA", "SHY_AMMA", "BIL_AMMA", "SLV_AMMA", "GLD_AMMA",
    "SPY_AMMA", "INDA_AMMA", "MCHI_AMMA", "IBIT_AMMA", "ETHA_AMMA", "EZU_AMMA",
]

# ── Load data ─────────────────────────────────────────────────
_raw = read_data(include_supplemental=True)
_ms, _prices = create_model_state(
    raw_data_bundle=_raw,
    features=list(FEATURES.keys()),
    start_date=_start,
    end_date=_end,
    universe=_universe,
)
print("Data loaded.")

# ── Register hybrid as a trading-engine model ─────────────────
# Uses hyb_w (precomputed regime-switched weights) from the cell above.
# Melted to long (date, ticker, weight) at construction time so no
# type-mismatched join is needed inside run_model.
def HybridRLModel(hybrid_weights_pd: pd.DataFrame):
    hw = hybrid_weights_pd.copy().reset_index()
    hw["date"] = pd.to_datetime(hw["date"]).dt.strftime("%Y-%m-%d")
    ticker_cols = [c for c in hw.columns if c != "date"]
    hw_long = (
        pl.from_pandas(hw)
        .with_columns(pl.col("date").str.to_date())
        .unpivot(index="date", on=ticker_cols,
                 variable_name="ticker", value_name="weight")
    )
    def run_model(lf: pl.LazyFrame) -> pl.LazyFrame:
        return hw_long.lazy()
    return run_model

# Build hybrid weight matrix: regime==1 → BMOM, regime==0 → RL
hyb_w = oos_w.copy()
for _date, _r in regime.items():
    if _r == 1 and _date in bmom_w_oos.index:
        hyb_w.loc[_date] = bmom_w_oos.loc[_date]

_candidate_registry = MODELS.copy()
_candidate_registry["hybrid_rl_global_equity"] = {
    "tickers": trade_tickers,
    "columns": [],
    "function": HybridRLModel(hyb_w[trade_tickers]),
}

# ── Full MVO pipeline ─────────────────────────────────────────
def run_pipeline(registry, models_list):
    ins = orchestrate_model_backtests(
        model_state_bundle=_ms,
        models=models_list,
        universe=_universe,
        registry=registry,
    )
    sims = orchestrate_model_simulations(
        prices=_prices, model_insights=ins,
        start_date=_start, end_date=_end,
    )
    agg = orchestrate_portfolio_aggregation(
        model_insights=ins, backtest_results=sims,
        universe=_universe, aggregators=_aggregators,
        start_date=_start, end_date=_end,
    )
    opt = orchestrate_portfolio_optimizations(
        prices=_prices, aggregated_insights=agg,
        universe=_universe, optimizers=_optimizers,
    )
    return orchestrate_portfolio_simulations(
        prices=_prices, portfolio_insights=opt,
        start_date=_start, end_date=_end,
        initial_value=_init_val,
    )

print("Running baseline (AMMA only)...")
_base_sim  = run_pipeline(MODELS, _prod_models)

print("Running candidate (AMMA + Hybrid through MVO)...")
_cand_sim  = run_pipeline(_candidate_registry,
                           _prod_models + ["hybrid_rl_global_equity"])

# ── Extract returns ───────────────────────────────────────────
def _get_ret(sim):
    df = sim["mean_variance_constrained"]["backtest_results"]
    if hasattr(df, "collect"): df = df.collect()
    df = df.to_pandas()
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date")["daily_return"].astype(float).sort_index()

base_ret = _get_ret(_base_sim)
cand_ret = _get_ret(_cand_sim)
marg_ret = (cand_ret - base_ret).dropna()

# ── Performance helpers ───────────────────────────────────────
ANN = 252
def _perf(r, label):
    r   = r.dropna()
    ar  = r.mean() * ANN;  av = r.std() * np.sqrt(ANN)
    sh  = ar / av if av > 0 else np.nan
    cum = (1 + r).cumprod()
    dd  = ((cum / cum.cummax()) - 1).min()
    neg = r[r < 0]
    ds  = neg.std() * np.sqrt(ANN) if len(neg) else np.nan
    so  = ar / ds if ds and ds > 0 else np.nan
    cal = ar / abs(dd) if dd < 0 else np.nan
    return {"Strategy": label,
            "Ann Ret (%)": round(ar*100,2), "Ann Vol (%)": round(av*100,2),
            "Sharpe": round(sh,3), "Max DD (%)": round(dd*100,2),
            "Calmar": round(cal,3), "Sortino": round(so,3)}

s_base = _perf(base_ret, "Baseline (AMMA)")
s_cand = _perf(cand_ret, "Candidate (AMMA + Hybrid via MVO)")
s_hyb  = _perf(hybrid_ret.reindex(base_ret.index).dropna(), "Hybrid Standalone")
s_marg = _perf(marg_ret,  "Marginal (Candidate − Baseline)")

corr_hyb = hybrid_ret.reindex(base_ret.index).dropna().corr(base_ret)
beta_hyb = (np.cov(hybrid_ret.reindex(base_ret.index).dropna(), base_ret)[0,1]*ANN) / (base_ret.var()*ANN)

# ── Model review ──────────────────────────────────────────────
print("\n" + "=" * 65)
print("MODEL REVIEW — Hybrid RL Global Equity Allocator")
print("  Baseline:  AMMA production portfolio (MVO)")
print("  Candidate: AMMA + Hybrid fed through MVO")
print("=" * 65)

print("\n" + pd.DataFrame([s_base, s_cand, s_hyb, s_marg]).to_string(index=False))
print(f"\n  Hybrid ↔ Baseline correlation: {corr_hyb:.3f}")
print(f"  Hybrid beta to Baseline:        {beta_hyb:.3f}")

c1 = s_hyb["Sharpe"] >= 0.7 or s_cand["Sharpe"] > s_base["Sharpe"]
c2 = s_cand["Ann Ret (%)"] > s_base["Ann Ret (%)"]
c3 = s_cand["Max DD (%)"]  > s_base["Max DD (%)"]
c4 = abs(corr_hyb) < 0.6
c5 = True
c6 = s_hyb["Sortino"] > 0 and s_cand["Sharpe"] > s_base["Sharpe"]

criteria = [
    ("1. Sharpe Improvement",          c1,
     f"Hybrid SA {s_hyb['Sharpe']:.3f} | Portfolio {s_base['Sharpe']:.3f} → {s_cand['Sharpe']:.3f}"),
    ("2. Return Improvement",          c2,
     f"Portfolio {s_base['Ann Ret (%)']:.2f}% → {s_cand['Ann Ret (%)']:.2f}%"),
    ("3. Drawdown Improvement",        c3,
     f"Portfolio {s_base['Max DD (%)']:.2f}% → {s_cand['Max DD (%)']:.2f}%  (less negative = better)"),
    ("4. Correlation / Diversification", c4,
     f"Hybrid ↔ Baseline ρ={corr_hyb:.3f}  β={beta_hyb:.3f}"),
    ("5. Economic Rationale",          c5,
     "PPO on macro momentum signals; BMOM regime switch for drawdown protection"),
    ("6. Transaction Costs / Net",     c6,
     f"Hybrid Sortino: {s_hyb['Sortino']:.3f} | Portfolio Sharpe improves: {c6}"),
]

score = sum(1 for _, p, _ in criteria if p)
print("\n--- Scorecard ---")
for name, passed, note in criteria:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {name}")
    print(f"          {note}")

verdict = ("RECOMMENDED FOR INCLUSION"   if score >= 3 else
           "BORDERLINE — FURTHER REVIEW" if score == 2 else
           "NOT RECOMMENDED")
print(f"\n  Criteria met: {score} / {len(criteria)}   →   {verdict}")
print("=" * 65)

# ── Equity curves ─────────────────────────────────────────────
pct2x = FuncFormatter(lambda y, _: f"{y:.2f}x")
fig, axes = plt.subplots(3, 1, figsize=(13, 13), sharex=True)
fig.suptitle("Hybrid RL — Model Review (MVO Portfolio)", fontsize=13, fontweight="bold")

ax = axes[0]
(1 + base_ret).cumprod().plot(ax=ax, color="#888888", linewidth=1.4, linestyle="--", label="Baseline (AMMA)")
(1 + hybrid_ret.reindex(base_ret.index).dropna()).cumprod().plot(
    ax=ax, color="#1f78b4", linewidth=1.6, linestyle="--", label="Hybrid Standalone")
(1 + cand_ret).cumprod().plot(ax=ax, color="#2ca02c", linewidth=1.8, label="Candidate (AMMA + Hybrid via MVO)")
ax.set_title("View 1 — Standalone vs MVO Portfolio", fontweight="bold")
ax.set_ylabel("Cumulative Return"); ax.yaxis.set_major_formatter(pct2x)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
(1 + base_ret).cumprod().plot(ax=ax, color="#888888", linewidth=1.4, linestyle="--", label="Baseline (without Hybrid)")
(1 + cand_ret).cumprod().plot(ax=ax, color="#2ca02c", linewidth=1.8, label="Candidate (with Hybrid via MVO)")
ax.set_title("View 2 — Portfolio With vs Without Hybrid (MVO)", fontweight="bold")
ax.set_ylabel("Cumulative Return"); ax.yaxis.set_major_formatter(pct2x)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[2]
marg_nav = (1 + marg_ret).cumprod()
marg_nav.plot(ax=ax, color="#2ca02c", linewidth=1.8, label="Marginal (Candidate − Baseline)")
ax.axhline(1.0, color="black", linewidth=0.8, linestyle="--")
ax.fill_between(marg_nav.index, marg_nav.values, 1.0,
                where=(marg_nav.values >= 1.0), alpha=0.12, color="#2ca02c")
ax.fill_between(marg_nav.index, marg_nav.values, 1.0,
                where=(marg_nav.values < 1.0),  alpha=0.12, color="#d62728")
ax.set_title("View 3 — Marginal Equity Curve", fontweight="bold")
ax.set_ylabel("Marginal Index"); ax.yaxis.set_major_formatter(pct2x)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
